In [1]:
import json
import pandas as pd
from collections.abc import Mapping, Sequence

with open("ipl_json/1527674.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# start from the innings list (keeps original nesting)
df = pd.DataFrame(data["innings"])

def explode_and_normalize_all(df: pd.DataFrame) -> pd.DataFrame:
    """
    Repeatedly explode the first column that contains lists (non-string),
    then normalize any dicts produced into columns. Continue until no list
    cells remain. Finally expand any remaining dict columns.
    """
    def is_list_cell(x):
        return isinstance(x, Sequence) and not isinstance(x, (str, bytes, bytearray, dict))
    # explode list columns until none remain
    while True:
        list_cols = [c for c in df.columns if df[c].apply(is_list_cell).any()]
        if not list_cols:
            break
        col = list_cols[0]
        df = df.explode(col).reset_index(drop=True)
        # if exploded values are dicts, expand them into prefixed columns
        if df[col].apply(lambda x: isinstance(x, dict)).any():
            norm = pd.json_normalize(df[col].fillna({}).tolist())
            if not norm.empty:
                norm = norm.add_prefix(f"{col}_")
                df = df.drop(columns=[col]).reset_index(drop=True).join(norm)
            else:
                df = df.drop(columns=[col])
    # expand any remaining dict columns
    dict_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, dict)).any()]
    for col in dict_cols:
        norm = pd.json_normalize(df[col].fillna({}).tolist())
        if not norm.empty:
            norm = norm.add_prefix(f"{col}_")
            df = df.drop(columns=[col]).reset_index(drop=True).join(norm)
        else:
            df = df.drop(columns=[col])
    return df

df_expanded = explode_and_normalize_all(df)
print("rows:", df_expanded.shape[0], "cols:", df_expanded.shape[1])
df_expanded.head()

rows: 225 cols: 28


,team,overs_over,powerplays_from,powerplays_to,powerplays_type,overs_deliveries_batter,overs_deliveries_bowler,overs_deliveries_non_striker,overs_deliveries_runs.batter,overs_deliveries_runs.extras,...,overs_deliveries_extras.byes,overs_deliveries_wickets_player_out,overs_deliveries_wickets_kind,overs_deliveries_replacements.match_in,overs_deliveries_replacements.match_out,overs_deliveries_replacements.match_team,overs_deliveries_replacements.match_reason,overs_deliveries_wickets_fielders_name,target_overs,target_runs
0,Sunrisers Hyderabad,0,0.1,5.6,mandatory,TM Head,JA Duffy,Abhishek Sharma,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sunrisers Hyderabad,0,0.1,5.6,mandatory,TM Head,JA Duffy,Abhishek Sharma,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Sunrisers Hyderabad,0,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sunrisers Hyderabad,0,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,6,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Sunrisers Hyderabad,0,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
df_expanded = explode_and_normalize_all(df)

# detect the over column (common names produced by the explode/normalize logic)
over_col = next((c for c in df_expanded.columns if c in ("overs_over", "over") or c.endswith("_over")), None)
# detect an innings identifier (team / inning column)
team_candidates = ["team", "innings_team", "team_name", "inning_team"]
team_col = next((c for c in team_candidates if c in df_expanded.columns), None)

# fallback: try to match a column containing one of the teams from the original JSON
if team_col is None:
    teams = [inning.get("team") for inning in data.get("innings", []) if "team" in inning]
    for c in df_expanded.columns:
        if df_expanded[c].dropna().isin(teams).any():
            team_col = c
            break

# compute ball_number (1-based): restart per innings if team_col found, otherwise per over or global
if over_col is None:
    df_expanded["ball_number"] = range(1, len(df_expanded) + 1)
else:
    if team_col:
        df_expanded["ball_number"] = df_expanded.groupby([team_col, over_col]).cumcount() + 1
    else:
        df_expanded["ball_number"] = df_expanded.groupby(over_col).cumcount() + 1

# move ball_number to immediate right of the over column (if both exist)
if over_col in df_expanded.columns and "ball_number" in df_expanded.columns:
    cols = list(df_expanded.columns)
    try:
        over_idx = cols.index(over_col)
        # remove ball_number then insert right after over_col
        cols.remove("ball_number")
        cols.insert(over_idx + 1, "ball_number")
        df_expanded = df_expanded[cols]
    except ValueError:
        # if something unexpected happens, leave column order unchanged
        pass

print("using over_col =", over_col, "team_col =", team_col)
print("rows:", df_expanded.shape[0], "cols:", df_expanded.shape[1])
df_expanded.head()

using over_col = overs_over team_col = team
rows: 225 cols: 29


,team,overs_over,ball_number,powerplays_from,powerplays_to,powerplays_type,overs_deliveries_batter,overs_deliveries_bowler,overs_deliveries_non_striker,overs_deliveries_runs.batter,...,overs_deliveries_extras.byes,overs_deliveries_wickets_player_out,overs_deliveries_wickets_kind,overs_deliveries_replacements.match_in,overs_deliveries_replacements.match_out,overs_deliveries_replacements.match_team,overs_deliveries_replacements.match_reason,overs_deliveries_wickets_fielders_name,target_overs,target_runs
0,Sunrisers Hyderabad,0,1,0.1,5.6,mandatory,TM Head,JA Duffy,Abhishek Sharma,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sunrisers Hyderabad,0,2,0.1,5.6,mandatory,TM Head,JA Duffy,Abhishek Sharma,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Sunrisers Hyderabad,0,3,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sunrisers Hyderabad,0,4,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Sunrisers Hyderabad,0,5,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df_expanded = df_expanded.rename(columns={"overs_over": "over"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_batter": "striker"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_bowler": "bowler"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_non_striker": "non_striker"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_runs.batter": "batter_runs"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_runs.extras": "extra_runs"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_runs.total": "total_runs"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_extras.wides": "wides"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_extras.legbyes": "legbyes"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_extras.noballs": "noballs"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_review.by": "review_by"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_review.umpire": "review_umpire"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_review.batter": "review_batter"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_review.decision": "review_decision"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_review.type": "review_type"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_review.umpires_call": "umpires_call"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_wickets_player_out": "player_out"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_wickets_kind": "wicket_kind"})
df_expanded = df_expanded.rename(columns={"overs_deliveries_wickets_fielders_name": "fielders_name"})

df_expanded.head(10)


,team,over,ball_number,powerplays_from,powerplays_to,powerplays_type,striker,bowler,non_striker,batter_runs,...,overs_deliveries_extras.byes,player_out,wicket_kind,overs_deliveries_replacements.match_in,overs_deliveries_replacements.match_out,overs_deliveries_replacements.match_team,overs_deliveries_replacements.match_reason,fielders_name,target_overs,target_runs
0,Sunrisers Hyderabad,0,1,0.1,5.6,mandatory,TM Head,JA Duffy,Abhishek Sharma,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sunrisers Hyderabad,0,2,0.1,5.6,mandatory,TM Head,JA Duffy,Abhishek Sharma,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Sunrisers Hyderabad,0,3,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sunrisers Hyderabad,0,4,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Sunrisers Hyderabad,0,5,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Sunrisers Hyderabad,0,6,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Sunrisers Hyderabad,1,1,0.1,5.6,mandatory,TM Head,B Kumar,Abhishek Sharma,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Sunrisers Hyderabad,1,2,0.1,5.6,mandatory,Abhishek Sharma,B Kumar,TM Head,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Sunrisers Hyderabad,1,3,0.1,5.6,mandatory,TM Head,B Kumar,Abhishek Sharma,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Sunrisers Hyderabad,1,4,0.1,5.6,mandatory,Abhishek Sharma,B Kumar,TM Head,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
team_candidates = ["team", "innings_team", "team_name", "inning_team"]
team_col = next((c for c in team_candidates if c in df_expanded.columns), None)
if team_col is None:
    teams = [inning.get("team") for inning in data.get("innings", []) if "team" in inning]
    for c in df_expanded.columns:
        if df_expanded[c].dropna().isin(teams).any():
            team_col = c
            break

# ensure total_runs exists and is numeric
if "total_runs" not in df_expanded.columns:
    raise KeyError("total_runs column not found in df_expanded")

df_expanded["total_runs"] = pd.to_numeric(df_expanded["total_runs"], errors="coerce").fillna(0).astype(int)

# cumulative sum per innings (resets when innings/team changes)
if team_col:
    df_expanded["current_team_total"] = df_expanded.groupby(team_col)["total_runs"].cumsum()
else:
    # fallback: global cumulative sum if no innings identifier found
    df_expanded["current_team_total"] = df_expanded["total_runs"].cumsum()

df_expanded.head(10)

,team,over,ball_number,powerplays_from,powerplays_to,powerplays_type,striker,bowler,non_striker,batter_runs,...,player_out,wicket_kind,overs_deliveries_replacements.match_in,overs_deliveries_replacements.match_out,overs_deliveries_replacements.match_team,overs_deliveries_replacements.match_reason,fielders_name,target_overs,target_runs,current_team_total
0,Sunrisers Hyderabad,0,1,0.1,5.6,mandatory,TM Head,JA Duffy,Abhishek Sharma,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,Sunrisers Hyderabad,0,2,0.1,5.6,mandatory,TM Head,JA Duffy,Abhishek Sharma,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,Sunrisers Hyderabad,0,3,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,Sunrisers Hyderabad,0,4,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
4,Sunrisers Hyderabad,0,5,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
5,Sunrisers Hyderabad,0,6,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7
6,Sunrisers Hyderabad,1,1,0.1,5.6,mandatory,TM Head,B Kumar,Abhishek Sharma,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8
7,Sunrisers Hyderabad,1,2,0.1,5.6,mandatory,Abhishek Sharma,B Kumar,TM Head,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9
8,Sunrisers Hyderabad,1,3,0.1,5.6,mandatory,TM Head,B Kumar,Abhishek Sharma,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10
9,Sunrisers Hyderabad,1,4,0.1,5.6,mandatory,Abhishek Sharma,B Kumar,TM Head,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11


In [5]:
df_expanded.to_csv("expanded_match_data.csv", index=False)

In [6]:
from pathlib import Path
import json
import pandas as pd
from collections.abc import Mapping, Sequence

# list the files you want to convert (edit as needed)
files = sorted(Path("ipl_json").glob("152*.json"))

out_dir = Path("csv_output")
out_dir.mkdir(parents=True, exist_ok=True)

# optional rename map (adjust if you changed earlier mappings)
rename_map = {
    "overs_over": "over",
    "overs_deliveries_batter": "striker",
    "overs_deliveries_bowler": "bowler",
    "overs_deliveries_non_striker": "non_striker",
    "overs_deliveries_runs.batter": "batter_runs",
    "overs_deliveries_runs.extras": "extra_runs",
    "overs_deliveries_runs.total": "total_runs",
    "overs_deliveries_extras.wides": "wides",
    "overs_deliveries_extras.legbyes": "legbyes",
    "overs_deliveries_extras.noballs": "noballs",
    "overs_deliveries_extras.byes": "byes",
    "overs_deliveries_review.by": "review_by",
    "overs_deliveries_review.umpire": "review_umpire",
    "overs_deliveries_review.batter": "review_batter",
    "overs_deliveries_review.decision": "review_decision",
    "overs_deliveries_review.type": "review_type",
    "overs_deliveries_review.umpires_call": "umpires_call",
    "overs_deliveries_wickets_player_out": "player_out",
    "overs_deliveries_wickets_kind": "wicket_kind",
    "overs_deliveries_wickets_fielders_name": "fielders_name",
}

def explode_and_normalize_all(df: pd.DataFrame) -> pd.DataFrame:
    """
    Repeatedly explode the first column that contains lists (non-string),
    then normalize any dicts produced into columns. Continue until no list
    cells remain. Finally expand any remaining dict columns.
    """
    def is_list_cell(x):
        return isinstance(x, Sequence) and not isinstance(x, (str, bytes, bytearray, dict))
    # explode list columns until none remain
    while True:
        list_cols = [c for c in df.columns if df[c].apply(is_list_cell).any()]
        if not list_cols:
            break
        col = list_cols[0]
        df = df.explode(col).reset_index(drop=True)
        # if exploded values are dicts, expand them into prefixed columns
        if df[col].apply(lambda x: isinstance(x, dict)).any():
            norm = pd.json_normalize(df[col].fillna({}).tolist())
            if not norm.empty:
                norm = norm.add_prefix(f"{col}_")
                df = df.drop(columns=[col]).reset_index(drop=True).join(norm)
            else:
                df = df.drop(columns=[col])
    # expand any remaining dict columns
    dict_cols = [c for c in df.columns if df[c].apply(lambda x: isinstance(x, dict)).any()]
    for col in dict_cols:
        norm = pd.json_normalize(df[col].fillna({}).tolist())
        if not norm.empty:
            norm = norm.add_prefix(f"{col}_")
            df = df.drop(columns=[col]).reset_index(drop=True).join(norm)
        else:
            df = df.drop(columns=[col])
    return df

for p in files:
    if not p.exists():
        print("skipping (not found):", p)
        continue
    try:
        with p.open("r", encoding="utf-8") as f:
            data = json.load(f)

        # base dataframe from innings
        df = pd.DataFrame(data.get("innings", []))

        # explode/normalize using the helper defined earlier in the notebook
        df_expanded = explode_and_normalize_all(df)

        # apply safe renames
        to_rename = {k: v for k, v in rename_map.items() if k in df_expanded.columns}
        if to_rename:
            df_expanded = df_expanded.rename(columns=to_rename)

        # detect team & over columns (same logic as earlier cells)
        team_candidates = ["team", "innings_team", "team_name", "inning_team"]
        team_col = next((c for c in team_candidates if c in df_expanded.columns), None)
        if team_col is None:
            teams = [inning.get("team") for inning in data.get("innings", []) if "team" in inning]
            for c in df_expanded.columns:
                if df_expanded[c].dropna().isin(teams).any():
                    team_col = c
                    break

        over_col = next((c for c in df_expanded.columns if c in ("overs_over", "over") or c.endswith("_over")), None)

        # compute 1-based ball_number (restart per innings) and position it after over_col
        if over_col is None:
            df_expanded["ball_number"] = range(1, len(df_expanded) + 1)
        else:
            if team_col:
                df_expanded["ball_number"] = df_expanded.groupby([team_col, over_col]).cumcount() + 1
            else:
                df_expanded["ball_number"] = df_expanded.groupby(over_col).cumcount() + 1
            if over_col in df_expanded.columns:
                cols = list(df_expanded.columns)
                try:
                    over_idx = cols.index(over_col)
                    cols.remove("ball_number")
                    cols.insert(over_idx + 1, "ball_number")
                    df_expanded = df_expanded[cols]
                except ValueError:
                    pass

        # compute current_team_total from total_runs (reset per innings)
        if "total_runs" in df_expanded.columns:
            df_expanded["total_runs"] = pd.to_numeric(df_expanded["total_runs"], errors="coerce").fillna(0).astype(int)
            if team_col:
                df_expanded["current_team_total"] = df_expanded.groupby(team_col)["total_runs"].cumsum()
            else:
                df_expanded["current_team_total"] = df_expanded["total_runs"].cumsum()
        else:
            df_expanded["current_team_total"] = 0

        # write one CSV per JSON input
        out_path = out_dir / (p.stem + ".csv")
        df_expanded.to_csv(out_path, index=False)
        print("saved:", out_path)
    except Exception as e:
        print("failed:", p, "->", e)
# ...existing code...

saved: csv_output/1527674.csv
saved: csv_output/1527675.csv
saved: csv_output/1527676.csv
saved: csv_output/1527677.csv
saved: csv_output/1527678.csv
saved: csv_output/1527679.csv
saved: csv_output/1527680.csv
saved: csv_output/1527681.csv
saved: csv_output/1527682.csv
saved: csv_output/1527683.csv
saved: csv_output/1527684.csv
saved: csv_output/1527685.csv
saved: csv_output/1527686.csv
saved: csv_output/1527687.csv
saved: csv_output/1527688.csv
saved: csv_output/1527689.csv
saved: csv_output/1527690.csv
saved: csv_output/1527691.csv
saved: csv_output/1527692.csv
saved: csv_output/1527693.csv
saved: csv_output/1529264.csv
saved: csv_output/1529265.csv
saved: csv_output/1529266.csv
saved: csv_output/1529267.csv
saved: csv_output/1529268.csv
saved: csv_output/1529269.csv
saved: csv_output/1529270.csv
saved: csv_output/1529271.csv
saved: csv_output/1529272.csv
saved: csv_output/1529273.csv
saved: csv_output/1529274.csv
saved: csv_output/1529275.csv
saved: csv_output/1529276.csv
saved: csv

In [7]:
csv_dir = Path("csv_output")
csv_files = sorted(csv_dir.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {csv_dir.resolve()}")

dfs = []
for p in csv_files:
    try:
        df = pd.read_csv(p)
        df["source_file"] = p.name  # optional: keep track of origin
        dfs.append(df)
    except Exception as e:
        print(f"warning: failed to read {p.name} -> {e}")

# concatenate into a single DataFrame
all_matches_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

# remove second of back-to-back identical run-out readings (within same source_file)
all_matches_df["player_out"] = all_matches_df["player_out"].fillna("").astype(str)
wicket_norm = all_matches_df["wicket_kind"].fillna("").astype(str).str.strip().str.lower()

same_source = all_matches_df["source_file"] == all_matches_df["source_file"].shift(1)
same_player_out = all_matches_df["player_out"] == all_matches_df["player_out"].shift(1)
prev_runout = wicket_norm.shift(1) == "run out"
curr_runout = wicket_norm == "run out"

consecutive_duplicate_runout = same_source & same_player_out & prev_runout & curr_runout

if consecutive_duplicate_runout.any():
    all_matches_df = all_matches_df[~consecutive_duplicate_runout].reset_index(drop=True)

# all_matches_df["venue"] = all_matches_df["source_file"].map(venue_map).astype("object")
# all_matches_df["avg_first_innings_score"] = pd.to_numeric(all_matches_df["venue"].map(avg_first_innings_map), errors="coerce")

# quick sanity checks
print("files read:", len(dfs), "rows total:", len(all_matches_df))
all_matches_df.head()

files read: 70 rows total: 16561


,team,over,ball_number,powerplays_from,powerplays_to,powerplays_type,striker,bowler,non_striker,batter_runs,...,source_file,noballs,overs_deliveries_wickets_fielders_substitute,umpires_call,overs_deliveries_replacements.role_in,overs_deliveries_replacements.role_out,overs_deliveries_replacements.role_reason,overs_deliveries_replacements.role_role,overs_deliveries_runs.non_boundary,super_over
0,Sunrisers Hyderabad,0,1.0,0.1,5.6,mandatory,TM Head,JA Duffy,Abhishek Sharma,0,...,1527674.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sunrisers Hyderabad,0,2.0,0.1,5.6,mandatory,TM Head,JA Duffy,Abhishek Sharma,1,...,1527674.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Sunrisers Hyderabad,0,3.0,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,0,...,1527674.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sunrisers Hyderabad,0,4.0,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,6,...,1527674.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Sunrisers Hyderabad,0,5.0,0.1,5.6,mandatory,Abhishek Sharma,JA Duffy,TM Head,0,...,1527674.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
all_matches_df.to_csv('all_matches.csv', index=False)

In [9]:
cleaned_all_matches_df = all_matches_df.copy()

cleaned_all_matches_df = cleaned_all_matches_df.drop(columns=["overs_deliveries_replacements.role_reason"], errors="ignore")
cleaned_all_matches_df = cleaned_all_matches_df.drop(columns=["overs_deliveries_replacements.role_role"], errors="ignore")

cleaned_all_matches_df = cleaned_all_matches_df.rename(columns={"overs_deliveries_replacements.match_in": "subbed_in"})
cleaned_all_matches_df = cleaned_all_matches_df.rename(columns={"overs_deliveries_replacements.match_out": "subbed_out"})
cleaned_all_matches_df = cleaned_all_matches_df.rename(columns={"overs_deliveries_replacements.match_team": "subbed_team"})
cleaned_all_matches_df = cleaned_all_matches_df.rename(columns={"overs_deliveries_replacements.match_reason": "subbed_reason"})
cleaned_all_matches_df = cleaned_all_matches_df.rename(columns={"overs_deliveries_wickets_fielders_substitute": "is_fielder_sub"})
cleaned_all_matches_df = cleaned_all_matches_df.rename(columns={"overs_deliveries_replacements.role_in": "injury_subbed_in"})
cleaned_all_matches_df = cleaned_all_matches_df.rename(columns={"overs_deliveries_replacements.role_out": "injury_subbed_out"})
cleaned_all_matches_df = cleaned_all_matches_df.rename(columns={"overs_deliveries_runs.non_boundary": "is_non_boundary"})

# Filter out super_over rows and drop the column
cleaned_all_matches_df = cleaned_all_matches_df[cleaned_all_matches_df["super_over"] != True].reset_index(drop=True)
cleaned_all_matches_df = cleaned_all_matches_df.drop(columns=["super_over"], errors="ignore")

print(f"Rows after filtering super_over: {len(all_matches_df)}")

Rows after filtering super_over: 16561


In [10]:
cleaned_all_matches_df.to_csv('cleaned_all_matches.csv', index=False)